# 🎓 TeachRL v2 — Training Notebook

**Theme 4: Self-Improvement** | Meta PyTorch Hackathon x Scaler 2025

This notebook trains a PPO agent on the TeachRL v2 environment using **HuggingFace TRL**.

The agent must:
1. **Identify** which of 8 hidden student archetypes it is teaching
2. **Adapt** its curriculum strategy in real-time
3. **Improve** as the self-play loop generates harder student variants

**Links:**
- HF Space: https://huggingface.co/spaces/ArchedEquation/TeachRL-v2
- GitHub: https://github.com/ArchedEquation/TeachRL-v2

---

## 1. Install Dependencies

In [ ]:
!pip install -q stable-baselines3 gymnasium pydantic fastapi uvicorn pyyaml openai wandb matplotlib

## 2. Clone Repository

In [ ]:
!git clone https://github.com/ArchedEquation/TeachRL-v2 /content/TeachRL
%cd /content/TeachRL
!ls

## 3. Verify Environment Works

In [ ]:
import sys
sys.path.insert(0, '/content/TeachRL')

from env.environment import TeachRLEnv, TASK_REGISTRY
from env.archetypes import ALL_ARCHETYPES

print(f'Tasks: {list(TASK_REGISTRY.keys())}')
print(f'Archetypes: {len(ALL_ARCHETYPES)}')

# Quick sanity check
env = TeachRLEnv(task_id='blind_teaching', seed=42, eval_mode=True)
obs = env.reset()
result = env.step({'concept': 'algebra_basics', 'difficulty': 'medium',
                   'hint_given': False, 'archetype_guess': None})
print(f'Step OK — reward={result.reward:.3f} done={result.done}')
print(f'True archetype: {env._sim.archetype_id.value}')
print(f'Expert hint: {obs.expert_hint[:60]}...')

## 4. Run Baseline Evaluation (Before Training)

In [ ]:
!python baseline/baseline_inference.py --episodes 10 --seed 42

## 5. (Optional) Login to W&B for Logging

In [ ]:
# Optional — skip if you don't want W&B logging
import wandb
wandb.login()

## 6. Train PPO — Easy Task (Archetype Identification)

In [ ]:
!python train_trl.py \
    --task archetype_identification \
    --steps 100000 \
    --seed 42 \
    --eval

## 7. Train PPO — Medium Task (Adaptive Curriculum)

In [ ]:
!python train_trl.py \
    --task adaptive_curriculum \
    --steps 200000 \
    --seed 42 \
    --eval

## 8. Train PPO — Hard Task (Blind Teaching)

In [ ]:
!python train_trl.py \
    --task blind_teaching \
    --steps 400000 \
    --seed 42 \
    --eval

## 9. Train PPO — Expert Task (Self-Play Escalation)

This is the **Theme 4 core task**. The environment gets harder as the agent improves.

In [ ]:
!python train_trl.py \
    --task self_play_escalation \
    --steps 500000 \
    --seed 42 \
    --eval

## 10. Full Evaluation — PPO vs All Baselines

In [ ]:
!python baseline/rl_agent.py --eval --task all --episodes 10 --seed 42

## 11. Display Training Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

plots = sorted(glob.glob('training_plots/*.png'))
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, plot_path in zip(axes.flat, plots[:4]):
    img = mpimg.imread(plot_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(plot_path.split('/')[-1].replace('.png','').replace('_',' ').title(),
                 fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('training_plots/all_plots_combined.png', dpi=150, bbox_inches='tight')
plt.show()
print('Combined plot saved!')

## 12. Demo — Watch the Trained Agent Teach

See how the PPO agent behaves vs a random agent on the hard task.

In [ ]:
from stable_baselines3 import PPO
from env.environment import TeachRLEnv
from env.gym_wrapper import obs_to_vector, int_to_action
from env.environment import TASK_REGISTRY
import numpy as np

task_id   = 'blind_teaching'
max_steps = TASK_REGISTRY[task_id]['max_steps']

try:
    model = PPO.load(f'models/ppo_{task_id}')

    def ppo_agent(obs_dict):
        vec = obs_to_vector(obs_dict, max_steps)
        action, _ = model.predict(vec, deterministic=True)
        c, d = int_to_action(int(action))
        return {'concept': c, 'difficulty': d, 'hint_given': False, 'archetype_guess': None}

    print('=== PPO Agent Demo ===')
    env = TeachRLEnv(task_id=task_id, seed=99, eval_mode=True)
    obs = env.reset(seed=99)
    print(f'True archetype: {env._sim.archetype_id.value}')
    print(f'Expert hint: {obs.expert_hint}')
    print()

    done = False
    while not done:
        action = ppo_agent(obs.model_dump())
        result = env.step(action)
        obs    = result.observation
        done   = result.done

    print(env.render())
    print(f'\nFinal Score: {env._task_score():.4f}')

except FileNotFoundError:
    print('No trained model found. Run cells 8 first.')

## 13. Test the Live API

The environment is deployed as a live API on HuggingFace Spaces.

In [ ]:
import requests, json

BASE_URL = 'https://archedequation-teachrl-v2.hf.space'

# Check health
r = requests.get(f'{BASE_URL}/')
print('Health:', r.json()['status'])
print('Tasks:', r.json()['tasks'])

# Start episode
r = requests.post(f'{BASE_URL}/reset',
                  json={'task_id': 'blind_teaching', 'seed': 42})
data = r.json()
sid  = data['session_id']
print(f'\nSession: {sid[:8]}...')
print(f'Task: {data["task"]["id"]} ({data["task"]["difficulty"]})')

# Take one step
r = requests.post(f'{BASE_URL}/step',
                  json={'session_id': sid, 'concept': 'algebra_basics',
                        'difficulty': 'medium', 'archetype_guess': 'anxious_perfectionist'})
step = r.json()
print(f'\nReward: {step["reward"]:.3f}  Done: {step["done"]}')
print(f'Task score: {step["info"]["task_score"]:.4f}')